# StockThink — render cinematics on a GPU

This renders the landing-page cinematics (e.g. the basement finale) at **high quality on a free GPU**, ~50× faster than a CPU box, then hands you the `.mp4` / `.webm` to drop into `frontend/public/landing/video/`.

**Before running:** `Runtime → Change runtime type → Hardware accelerator: GPU (T4)`.

You'll need the kit produced locally by `node editor/make-render-kit.mjs` — **`render-kit.zip`** (or `render-kit.tgz`), in `frontend/landing/`. Run the cells top to bottom; cell 3 prompts you to upload it.

In [ ]:
# 1) Confirm we actually got a GPU runtime
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || print('NO GPU — set Runtime > Change runtime type > GPU')

In [ ]:
# 2) System deps: Node 20, Google Chrome, ffmpeg
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get -qq install -y nodejs ffmpeg > /dev/null 2>&1
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get -qq install -y ./google-chrome-stable_current_amd64.deb > /dev/null 2>&1
!node --version && google-chrome --version && ffmpeg -version | head -1

In [ ]:
# 3) Upload the kit (render-kit.zip OR render-kit.tgz, from frontend/landing/), extract, install `ws`
from google.colab import files
up = files.upload()                       # choose render-kit.zip or render-kit.tgz
name = next(iter(up))
!rm -rf /content/render-kit
if name.endswith(('.tgz', '.tar.gz')):
    !tar -xzf "{name}" -C /content
else:
    !unzip -q -o "{name}" -d /content
!cd /content/render-kit && npm install --silent
print('kit ready')

In [ ]:
# 4) Serve the bundle statically (persists across cells)
import subprocess, time, urllib.request
srv = subprocess.Popen(['python3','-m','http.server','8000'], cwd='/content/render-kit',
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
code = urllib.request.urlopen('http://localhost:8000/record/studio.html?scene=ender').getcode()
print('static server on :8000  studio.html ->', code)

In [ ]:
# 5) RENDER on the GPU. Watch the printed 'GL renderer' line — it should say NVIDIA, not SwiftShader.
#    Bump W/H to 3840/2160 for 4K, or FRAMES for a longer/smoother clip — the GPU makes it cheap.
!cd /content/render-kit && \
  GL=gpu PORT=8000 URLPATH=/record/studio.html \
  SCENE=ender FRAMES=180 W=2560 H=1440 SS=1.5 OUT=/tmp/rec/ender \
  node editor/record.mjs

In [ ]:
# 6) Encode to mp4 (H.264) + webm (VP9) + poster
!cd /content/render-kit && \
  SCENE=ender FPS=30 CRF=17 IN=/tmp/rec/ender OUTDIR=/content/out \
  node editor/encode.mjs
!ls -la /content/out

In [ ]:
# 7) Download the finished clips. Put them in frontend/public/landing/video/ (replacing the old ones).
from google.colab import files
for f in ['ender.mp4','ender.webm','ender-poster.jpg']:
    files.download('/content/out/' + f)

## Notes
- **Verify the GPU is used:** cell 5 prints the WebGL renderer. If it says *SwiftShader*, the GPU fell back — re-check the GPU runtime; it'll still render, just slower.
- **Quality knobs (cell 5):** `W`/`H` (resolution), `SS` (supersample, 1–2), `FRAMES` (÷ `FPS` = seconds). `CRF` in cell 6 (lower = higher quality, bigger file).
- **Other scenes:** set `SCENE=coach` once the coach studio is wired (same flow).